# Synapse: AI-Native Engineering Mindset Challenge

Welcome to your first real-world AI-native engineering challenge! 

In this lab, we are going to bridge the gap between **traditional software engineering** and **AI engineering**. 

### 🎯 The Real-World Scenario
You have been assigned to build an **Automated Support Ticket Router**. 

**The Problem:** Your company receives thousands of support emails a day. Humans are too slow to sort them. You've been told to use an LLM (OpenAI) to classify every incoming ticket by **Department** and **Priority** so they can be routed to the right database.

### 🧠 The Engineering Narrative
By the end of this notebook, you will have navigated the three core tensions of AI engineering:
1.  **The Runtime Lens:** LLMs are slow and expensive. We cannot afford retries.
2.  **The "Chatty" Problem:** LLMs are conversational by nature. Even when you ask for JSON, they often add filler like *"Sure! Here is the JSON..."* which crashes standard Python code.
3.  **The Structured Solution:** Moving from brittle Regex boundaries to the modern industry standard: **OpenAI Structured Outputs**.

## 🛠️ Step 0: Resilient Setup

To run this lab, you need an **OpenAI API Key**.

In [ ]:
import getpass
import os
import time
import json
import re
from typing import List, Literal, Optional
from pydantic import BaseModel

try:
    from dotenv import load_dotenv
    load_dotenv()  # This loads the OPENAI_API_KEY from your .env file
    from openai import OpenAI, AuthenticationError, RateLimitError
except ImportError:
    !pip install -q openai python-dotenv pydantic
    from dotenv import load_dotenv
    load_dotenv()
    from openai import OpenAI, AuthenticationError, RateLimitError

# Securely input API key
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

try:
    client = OpenAI()
    client.models.list()
    print("✅ API Client initialized successfully!")
except AuthenticationError:
    print("❌ ERROR: Invalid API Key.")
except Exception as e:
    print(f"❌ ERROR: {e}")

## 🏁 Part 1: The API Contract & Runtime Lens

In AI engineering, the API call has two primary roles defined in the `messages` array:

1.  **`system` (The Rules of Engagement):** This is where the developer defines the "contract." You tell the model who it is, what rules to follow, and exactly what output format you expect. The end-user rarely sees this.
2.  **`user` (The Data to Process):** This is the actual messy, unpredictable customer input. 

### Tension 1: Retries are too expensive
Traditional functions take microseconds and are free. LLM calls take **seconds** and cost **money**. If your code crashes because the AI returned a bad format, you can't just run a loop to "try again"—it will kill your latency SLA and drain your budget.

Run the cell below to see a "Naive" implementation that summarizes a ticket.

In [ ]:
MESSY_TICKET = """
Subject: HELP!!! I can't log in and I'm losing money!!!
Body: Hey, I've been trying to get into my billing dashboard for three hours. 
It just says 'Server Error'. This is unacceptable. I have a big client meeting 
in an hour and I need to download my invoice. My account email is boss@startup.io. 
Fix this now or I'm cancelling my subscription!!
"""

def naive_summary(text):
    start_time = time.time()
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Summarize support tickets in one sentence."},
            {"role": "user", "content": text}
        ]
    )
    
    latency = time.time() - start_time
    usage = response.usage
    cost = (usage.prompt_tokens * 0.00000015) + (usage.completion_tokens * 0.00000060)
    
    print(f"Summary: {response.choices[0].message.content}")
    print(f"--- Runtime Stats ---\nLatency: {latency:.2f}s | Cost: ${cost:.6f}")

naive_summary(MESSY_TICKET)

## 🧨 Part 2: The "Chatty" AI Problem

In your real-world role, you've written a script to extract JSON from these tickets. It works in your local tests. But once you deploy it, you encounter the **Chatty Model** syndrome. 

LLMs are trained to be helpful assistants. When you ask for JSON, they often can't help themselves and add "filler" text or Markdown backticks. 

**Example Output:**
> *"Sure thing! I have analyzed the support ticket. Here is the classification in the JSON format you requested:"
> 
> ```json 
> { "department": "billing", "priority": "high" } 
> ``` 
> "I hope this helps!"*

If you pass this string directly to `json.loads()`, your application will **crash**. Run the cell below to see the reality of an AI-native crash.

In [ ]:
# A simulated LLM response with "Conversational Filler"
CHATTY_AI_RESPONSE = """
Sure! Based on the ticket, here is the routing information in JSON format:

```json
{
  "department": "billing",
  "priority": "high"
}
```
"""

try:
    # This will fail because the string contains non-JSON text
    data = json.loads(CHATTY_AI_RESPONSE)
    print("Success!", data)
except json.JSONDecodeError as e:
    print(f"❌ CRASH! JSONDecodeError: {e}")
    print("\nAI Engineering Tip: Never trust a raw LLM output to be pure JSON.")

## 🛡️ Part 3: The Challenge (Regex Boundaries)

How do we solve this? 

1. **Better Instructions:** You can try adding "Return ONLY raw JSON" to your prompt. (This helps, but doesn't guarantee 100% success).
2. **The Regex Boundary:** You can write Python code to surgically extract the JSON from the mess. 

**Your Task:** Implement a function that uses Regex to find the JSON block inside a chatty response. This is the classic "Defensive Boundary" every AI engineer used before 2024.

In [ ]:
def parse_ticket_with_regex(llm_output):
    """
    Use Regex to extract the JSON block from a potentially chatty response.
    Hint: Use re.search(r"(\{.*\})", llm_output, re.DOTALL)
    """
    # --- YOUR CODE HERE ---
    
    return None

print("Regex Result:", parse_ticket_with_regex(CHATTY_AI_RESPONSE))

## 🚀 Part 4: The AI-Native Solution (Structured Outputs)

Regex is brittle. If the model outputs two JSON objects or nested structures, your Regex might break. 

The modern industry standard is **OpenAI Structured Outputs**. Instead of parsing a string, you provide a **Pydantic Model** and OpenAI's engine uses **Constrained Decoding** to guarantee that the model *cannot* output anything but valid JSON that matches your schema.

### ⚠️ Capability Awareness: One Model Does Not Fit All

As an AI-Native Engineer, you must realize that **LLM features are not universal**. 

- **Model-Specific Features:** Structured Outputs (via `response_format`) is a specific capability. While `gpt-4o` supports it, older models like `gpt-3.5-turbo` or many open-source models only support a weaker "JSON Mode" (which ensures valid JSON but *not* schema adherence).
- **Capability Matrices:** Just because you've mastered a tool doesn't mean it works everywhere. Some models support **Function Calling**, others support **Reasoning** (like `o1`), and some are specialized only for **Embedding** or **Vision**. 
- **The Engineering Decision:** Part of your job is selecting the right model for the right capability. Don't pay for a "Reasoning" model if you only need a "Classifier."

**Your Task:** Use `client.chat.completions.parse` with the provided Pydantic model. 

Reference documentation: [OpenAI Structured Outputs Guide](https://developers.openai.com/api/docs/guides/structured-outputs?lang=python)

In [ ]:
class TicketRouting(BaseModel):
    department: Literal["billing", "technical", "sales"]
    priority: Literal["low", "medium", "high"]
    summary: str

def structured_routing_challenge(ticket_text):
    """
    IMPLEMENT THIS.
    Use client.chat.completions.parse() with the TicketRouting model.
    Handle the case where the model refuses to respond.
    """
    # completion = client.chat.completions.parse(
    #     model="gpt-4o-mini",
    #     messages=[
    #         {"role": "system", "content": "Classify the support ticket into department and priority."},
    #         {"role": "user", "content": ticket_text}
    #     ],
    #     response_format=TicketRouting,
    # )

    # # Access the parsed message
    # message = completion.choices[0].message
    
    # # --- YOUR CODE HERE: Handle Refusal vs Parsed Data ---
    
    return None

print("Testing Structured Outputs...")
# result = structured_routing_challenge(MESSY_TICKET)
# if result: print(f"Parsed: {result}")

## 📱 Return to Synapse

Congratulations! You've navigated the evolution from naive API calls to **Guaranteed Structured Outputs**. 

Scan the QR code below with your phone to jump directly back to the Synapse app.

In [ ]:
import qrcode
from IPython.display import display

NEXT_MODULE_URL = "https://ai-in-a-shell.com/learn/02-conceptual-llm-mechanics"

def generate_handoff_qr(url):
    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(url)
    qr.make(fit=True)
    img = qr.make_image(fill_color="black", back_color="white")
    display(img)

generate_handoff_qr(NEXT_MODULE_URL)